# 03 — Evaluate Metrics on KITTI-C Predictions

For each prediction:
1. Load raw MiDaS `.npy` prediction
2. Load ground-truth depth
3. Apply scale-and-shift alignment (**required** — MiDaS outputs relative inverse depth)
4. Compute Abs Rel, RMSE, δ1, etc.
5. Save results to `outputs/metrics/kittic_results.csv`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'configs' / 'dataset_paths.yaml').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root from the current working directory')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
MODEL_TYPE = 'dpt_hybrid_384'
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifests' / 'kitti_c_manifest.csv'
PRED_DIR = PROJECT_ROOT / 'outputs' / 'predictions' / 'kitti_c'
OUT_CSV = PROJECT_ROOT / 'outputs' / 'metrics' / 'kittic_results.csv'

MIN_DEPTH = 1e-3
MAX_DEPTH = 80.0

In [3]:
import numpy as np
import pandas as pd
import cv2
from tqdm.notebook import tqdm

from src.evaluation.align import align_scale_shift
from src.evaluation.metrics import compute_all_metrics
from src.datasets.transforms import load_kitti_depth
from src.utils.io import save_metrics_csv

df = pd.read_csv(MANIFEST_PATH, dtype={'frame_id': str})
df = df[df['gt_path'].notna()].reset_index(drop=True)

if len(df) == 0:
    raise RuntimeError(
        "No ground-truth depth paths found in the manifest. "
        "Download the KITTI depth GT (proj_depth/groundtruth layout) and place it at "
        f"{PROJECT_ROOT / 'data' / 'raw' / 'kitti_c'}, "
        "then re-run notebook 01 to rebuild the manifest with gt_path populated."
    )

print(f'Evaluating {len(df)} samples')


Evaluating 59332 samples


In [4]:
records = []


def prediction_path(row):
    return (
        PRED_DIR
        / row['corruption_type']
        / str(row['severity'])
        / row['sequence']
        / f"{row['frame_id']}.npy"
    )

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred_path = prediction_path(row)
    if not pred_path.exists():
        continue  # inference not yet run for this sample

    try:
        pred = np.load(str(pred_path))
        gt = load_kitti_depth(row['gt_path'])

        if gt.shape != pred.shape:
            gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]), interpolation=cv2.INTER_NEAREST)

        valid_mask = gt > 0
        aligned = align_scale_shift(pred, gt, valid_mask)
        m = compute_all_metrics(aligned, gt, valid_mask, MIN_DEPTH, MAX_DEPTH)

        records.append({
            'image_path': row['image_path'],
            'gt_path': row['gt_path'],
            'pred_path': str(pred_path),
            'dataset': 'kitti_c',
            'corruption_type': row['corruption_type'],
            'severity': row['severity'],
            'sequence': row['sequence'],
            'frame_id': row['frame_id'],
            'model_name': MODEL_TYPE,
            **m,
        })

    except Exception as e:
        print(f'ERROR: {row["image_path"]}: {e}')

results_df = pd.DataFrame(records)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUT_CSV, index=False)
print(f'Saved {len(results_df)} results to {OUT_CSV}')


  0%|          | 0/59332 [00:00<?, ?it/s]

Saved 59332 results to /home/kaiyul3/cs543/outputs/metrics/kittic_results.csv


In [5]:
from src.analysis.report_tables import corruption_summary_table

print('=== Mean metrics per corruption type ===')
corruption_summary_table(results_df)

=== Mean metrics per corruption type ===


,abs_rel,sq_rel,rmse,rmse_log,delta1,delta2,delta3
corruption_type,,,,,,,
clean,0.300600,1.992623,6.509561,1.302487,0.511118,0.796757,0.890634
brightness,0.301877,1.985665,6.520552,1.318311,0.507209,0.797672,0.891045
elastic_transform,0.304479,2.018514,6.543849,1.326134,0.502446,0.794640,0.889272
contrast,0.307479,2.067239,6.594641,1.330802,0.502485,0.791658,0.888442
pixelate,0.309759,2.080037,6.652829,1.364980,0.492550,0.790964,0.887144
fog,0.311602,2.164384,6.693770,1.227398,0.499945,0.785296,0.888319
color_quant,0.318598,2.225249,6.800846,1.268800,0.486455,0.778280,0.884824
shot_noise,0.322812,2.375472,7.027143,1.107281,0.487252,0.767766,0.886957
motion_blur,0.324435,2.355256,6.963136,1.213061,0.482125,0.774153,0.882890


## Next step

Run `04_failure_case_analysis.ipynb` to visualise worst/median/best failures.